# Lezione 4 – Pratica: Ottimizzazione I, come apprende una rete

Questo notebook è la pratica della Lezione 4 del corso *Advanced Deep Learning* (deep learning avanzato). Lo completi autonomamente, dopo la lezione in diretta e dopo aver letto il capitolo "Ottimizzazione I: come apprende una rete".

## Cosa farai in questa pratica

- Implementi da zero la regola di aggiornamento della discesa del gradiente (*gradient descent*, GD), l'algoritmo che regola i parametri di qualsiasi rete neurale per ridurne la perdita.
- Visualizzi come il tasso di apprendimento (*learning rate*) determina se il processo converge, converge lentamente o diverge su una superficie di perdita a forma di canyon stretto.
- Costruisci una funzione che prova diversi tassi di apprendimento candidati e sceglie automaticamente quello che funziona meglio, un caso d'uso tipico quando si regolano gli iperparametri.

## Prerequisito

Prima di iniziare, assicurati di aver seguito la Lezione 4 ("Ottimizzazione per *machine learning*", apprendimento automatico) e di aver letto il capitolo corrispondente del manuale. Devi capire cos'è un gradiente, cos'è il tasso di apprendimento e cos'è il numero di condizionamento di una funzione.

## Come lavorare con questo notebook

Ogni esercizio ha tre parti: il testo del caso, una cella di codice con parti contrassegnate come `# TODO` che completi tu, e una soluzione collassata subito sotto per verificare il tuo lavoro. **Prova a risolvere ogni esercizio da solo prima di aprire la soluzione.**

Esegui per prima cosa la cella delle importazioni.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## Esercizio 1. Implementa la discesa del gradiente da zero

### Il caso

Lavori nel team di data science di un'azienda che vuole capire, con un esempio minimo, come apprende una rete neurale prima di affidarsi alle librerie di deep learning (*deep learning*, DL) che automatizzano questo passo. Il tuo compito è costruire la funzione `descenso_de_gradiente`, il cuore di qualsiasi algoritmo di ottimizzazione basato sui gradienti.

Ricorda la regola di aggiornamento del capitolo:

$$\theta_{k+1} = \theta_k - \alpha \, \nabla h(\theta_k)$$

dove $\theta_k$ è il punto al passo $k$, $\alpha$ è il tasso di apprendimento e $\nabla h(\theta_k)$ è il gradiente della funzione obiettivo valutato in $\theta_k$. A ogni passo ti muovi in direzione opposta al gradiente: la direzione di massima crescita della perdita.

### Suggerimenti

- La funzione riceve la perdita e il gradiente come funzioni Python, quindi puoi chiamarle come `gradiente(punto_actual)`.
- Salva ogni punto visitato nella lista `trayectoria`, incluso il punto iniziale, per poter disegnare poi l'intero percorso.
- Non ti serve usare `funcion_perdida` dentro il ciclo di aggiornamento: la ricevi perché la userai nell'Esercizio 2.
- È già scritto un freno di sicurezza che interrompe il ciclo se i valori superano 1e6, per evitare overflow numerici quando il tasso di apprendimento è troppo alto.

In [ ]:
def descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa_aprendizaje, n_pasos):
    """
    Ejecuta el descenso de gradiente y devuelve la trayectoria completa de puntos visitados.

    Parámetros
    ----------
    funcion_perdida : función que recibe un array theta y devuelve un escalar.
    gradiente : función que recibe un array theta y devuelve el gradiente (mismo tamaño que theta).
    punto_inicial : lista o array con el punto de partida theta_0.
    tasa_aprendizaje : alpha, el tamaño del paso.
    n_pasos : número de iteraciones a ejecutar como máximo.

    Devuelve
    --------
    np.ndarray de forma (num_puntos, d) con cada punto visitado, incluido el inicial.
    """
    punto_actual = np.array(punto_inicial, dtype=float)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # TODO: completa la regla de actualización del descenso de gradiente.
        # theta_(k+1) = theta_k - alpha * gradiente(theta_k)
        nuevo_punto = ...

        punto_actual = nuevo_punto
        trayectoria.append(punto_actual.copy())

        # Freno de seguridad: si el proceso diverge, los valores pueden desbordar
        # el tipo float. Cortamos aquí para evitar errores numéricos.
        if np.any(np.abs(punto_actual) > 1e6):
            break

    return np.array(trayectoria)

In [ ]:
# Función de pérdida cuadrática de dos parámetros: h(theta) = 0.5 * (a * theta_1^2 + b * theta_2^2)
# 'a' y 'b' son las curvaturas en cada dirección. Con b mucho mayor que a,
# la superficie se parece al cañón estrecho de la analogía del capítulo.
a, b = 1.0, 20.0  # número de condición kappa = b / a = 20

def perdida_cuadratica(theta):
    return 0.5 * (a * theta[0] ** 2 + b * theta[1] ** 2)

def gradiente_cuadratica(theta):
    return np.array([a * theta[0], b * theta[1]])

punto_inicial = [4.0, 4.0]
tasa_aprendizaje = 0.04
n_pasos = 40

trayectoria = descenso_de_gradiente(perdida_cuadratica, gradiente_cuadratica, punto_inicial, tasa_aprendizaje, n_pasos)

# Dibujamos las curvas de nivel de la función y la trayectoria seguida
theta1_rango = np.linspace(-6, 6, 200)
theta2_rango = np.linspace(-6, 6, 200)
T1, T2 = np.meshgrid(theta1_rango, theta2_rango)
Z = 0.5 * (a * T1 ** 2 + b * T2 ** 2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(T1, T2, Z, levels=25, cmap="viridis", alpha=0.6)
ax.plot(trayectoria[:, 0], trayectoria[:, 1], marker="o", markersize=3, color="tab:red",
        label=f"Trayectoria (alpha={tasa_aprendizaje})")
ax.plot(0, 0, "k*", markersize=15, label="Mínimo")
ax.set_xlabel("theta_1 (dirección de baja curvatura)")
ax.set_ylabel("theta_2 (dirección de alta curvatura)")
ax.set_title("Trayectoria del descenso de gradiente")
ax.legend()
plt.show()

print(f"Pérdida final: {perdida_cuadratica(trayectoria[-1]):.6f}")
print(f"Número de puntos en la trayectoria: {len(trayectoria)}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
def descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa_aprendizaje, n_pasos):
    """
    Esegue la discesa del gradiente e restituisce la traiettoria completa dei punti visitati.
    """
    punto_actual = np.array(punto_inicial, dtype=float)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # Regola di aggiornamento: ci muoviamo in direzione opposta al gradiente,
        # cioè verso dove la funzione decresce più rapidamente.
        nuevo_punto = punto_actual - tasa_aprendizaje * gradiente(punto_actual)

        punto_actual = nuevo_punto
        trayectoria.append(punto_actual.copy())

        # Freno di sicurezza in caso di divergenza numerica.
        if np.any(np.abs(punto_actual) > 1e6):
            break

    return np.array(trayectoria)
```
</details>

## Esercizio 2. Trova automaticamente un buon tasso di apprendimento

### Il caso

Un collega addestra un modello su una funzione di perdita con un numero di condizionamento alto (un canyon molto più stretto di quello dell'Esercizio 1) e non sa quale tasso di apprendimento scegliere: se lo imposta troppo alto, l'addestramento diverge; se lo imposta troppo basso, impiega un'eternità. Ti chiede di automatizzare la ricerca: provare un elenco di tassi candidati e lasciare che sia il codice a decidere quale funziona meglio, senza che nessuno debba guardare i grafici uno per uno.

Questo è un caso d'uso reale: prima di lanciare un addestramento lungo e costoso, conviene esplorare diversi tassi di apprendimento in pochi passi e scartare subito quelli che divergono.

### Cosa completi

Completa la funzione `diverge`, che decide se una traiettoria della discesa del gradiente è sfuggita al controllo. Con questo pezzo, la funzione `buscar_mejor_tasa` (già scritta) scorre la lista dei tassi candidati, scarta quelli che divergono e tiene quello che ottiene la perdita finale minore.

### Suggerimenti

- Quando una traiettoria diverge, l'ultimo punto ha un valore di perdita enorme, o addirittura `inf` o `NaN` (dall'inglese *not a number*, non è un numero) se il tipo `float` va in overflow.
- `np.isfinite(valor)` ti dice se un valore è finito (`False` se è `inf` o `NaN`).
- Non serve controllare tutti i valori della traiettoria: basta guardare l'ultimo punto, perché se il processo diverge, la perdita cresce in modo incontrollato verso la fine.

In [ ]:
def diverge(trayectoria, funcion_perdida, umbral=1e6):
    """
    Decide si una trayectoria del descenso de gradiente ha divergido.

    Parámetros
    ----------
    trayectoria : array de puntos visitados, tal como lo devuelve descenso_de_gradiente.
    funcion_perdida : función que recibe un array theta y devuelve un escalar.
    umbral : valor de pérdida por encima del cual consideramos que el proceso se ha disparado.

    Devuelve
    --------
    True si la trayectoria ha divergido, False si se mantiene acotada.
    """
    ultimo_punto = trayectoria[-1]
    ultimo_valor = funcion_perdida(ultimo_punto)

    # TODO: decide el criterio de divergencia a partir de 'ultimo_valor' y 'umbral'.
    # Pista: una trayectoria ha divergido si su último valor de pérdida no es finito
    # (inf o NaN) o si supera el umbral.
    ha_divergido = ...

    return ha_divergido


def buscar_mejor_tasa(funcion_perdida, gradiente, punto_inicial, tasas_candidatas, n_pasos):
    """
    Prueba cada tasa de aprendizaje candidata, ejecuta el descenso de gradiente
    y devuelve la tasa que consigue la menor pérdida final sin diverger.

    Devuelve
    --------
    mejor_tasa : la tasa de aprendizaje candidata con menor pérdida final entre las que no divergen.
    resultados : diccionario {tasa: pérdida_final}, con np.inf para las tasas que divergen.
    """
    resultados = {}
    for tasa in tasas_candidatas:
        trayectoria = descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa, n_pasos)
        if diverge(trayectoria, funcion_perdida):
            resultados[tasa] = np.inf
        else:
            resultados[tasa] = funcion_perdida(trayectoria[-1])

    mejor_tasa = min(resultados, key=resultados.get)
    return mejor_tasa, resultados

In [ ]:
# Función de pérdida con un cañón mucho más estrecho que en el Ejercicio 1
a2, b2 = 1.0, 100.0  # numero de condicion kappa = 100

def perdida_valle(theta):
    return 0.5 * (a2 * theta[0] ** 2 + b2 * theta[1] ** 2)

def gradiente_valle(theta):
    return np.array([a2 * theta[0], b2 * theta[1]])

punto_inicial_valle = [4.0, 4.0]
n_pasos_valle = 50
tasas_candidatas = [0.0005, 0.001, 0.005, 0.008, 0.01, 0.015, 0.02, 0.05]

mejor_tasa, resultados = buscar_mejor_tasa(
    perdida_valle, gradiente_valle, punto_inicial_valle, tasas_candidatas, n_pasos_valle
)

for tasa, valor_final in sorted(resultados.items()):
    estado = "DIVERGE" if not np.isfinite(valor_final) else f"pérdida final = {valor_final:.6f}"
    print(f"alpha = {tasa:<8} -> {estado}")

L2 = max(a2, b2)
print(f"\nMejor tasa de aprendizaje encontrada: {mejor_tasa}")
print(f"Tasa máxima teórica de estabilidad (1/L): {1 / L2:.4f}")

In [ ]:
# Visualizamos la pérdida de cada tasa candidata a lo largo de las iteraciones
fig, ax = plt.subplots(figsize=(8, 6))
for tasa in tasas_candidatas:
    trayectoria = descenso_de_gradiente(perdida_valle, gradiente_valle, punto_inicial_valle, tasa, n_pasos_valle)
    valores = [perdida_valle(theta) for theta in trayectoria]
    estilo = "--" if diverge(trayectoria, perdida_valle) else "-"
    ax.plot(valores, estilo, marker="o", markersize=2, label=f"alpha={tasa}")

ax.set_yscale("log")
ax.set_xlabel("Iteración")
ax.set_ylabel("Pérdida [escala log]")
ax.set_title(f"Comparación de tasas de aprendizaje (kappa={b2/a2:.0f})")
ax.legend(fontsize=8)
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
def diverge(trayectoria, funcion_perdida, umbral=1e6):
    """
    Decide se una traiettoria della discesa del gradiente è divergita.
    """
    ultimo_punto = trayectoria[-1]
    ultimo_valor = funcion_perdida(ultimo_punto)

    # È divergita se l'ultimo valore di perdita non è finito (inf o NaN)
    # oppure se supera la soglia: in entrambi i casi il processo è sfuggito al controllo.
    ha_divergido = (not np.isfinite(ultimo_valor)) or (ultimo_valor > umbral)

    return ha_divergido


def buscar_mejor_tasa(funcion_perdida, gradiente, punto_inicial, tasas_candidatas, n_pasos):
    resultados = {}
    for tasa in tasas_candidatas:
        trayectoria = descenso_de_gradiente(funcion_perdida, gradiente, punto_inicial, tasa, n_pasos)
        if diverge(trayectoria, funcion_perdida):
            resultados[tasa] = np.inf
        else:
            resultados[tasa] = funcion_perdida(trayectoria[-1])

    mejor_tasa = min(resultados, key=resultados.get)
    return mejor_tasa, resultados
```
</details>

## Chiusura

Hai appena costruito, con NumPy puro e senza alcun framework di deep learning, l'algoritmo che rende possibile a qualsiasi rete neurale di apprendere. Quando nella prossima lezione userai `optimizer.step()` in PyTorch, saprai esattamente cosa succede dentro: una sottrazione tra il punto attuale e il gradiente scalato dal tasso di apprendimento, ripetuta migliaia di volte.

Hai anche verificato una cosa che ti accompagnerà per tutta la tua carriera nell'IA: non esiste un tasso di apprendimento universale. Dipende dalla curvatura del problema, e automatizzarne la ricerca, come nell'Esercizio 2, è una pratica comune prima di lanciare un addestramento lungo.

**Torna al tuo notebook, cambia il numero di condizionamento (il valore di `b2`) a 500 o 1000, e verifica quanto diventa più difficile trovare un tasso di apprendimento che non diverga.**